In [18]:
#shap[0] represent rows while shape[1] representcolumns



%pip install qdrant-client

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import os

load_dotenv()
URL = os.getenv("URL")
API_KEY = os.getenv("API_KEY")

model = SentenceTransformer("all-MiniLM-L6-v2")



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1596.64it/s]


In [23]:
qdrant_client = QdrantClient(
    url=URL,
    api_key=API_KEY
)

from qdrant_client.http.exceptions import UnexpectedResponse

try:
    qdrant_client.create_collection(
        collection_name="movies",
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )
    print("Collection created")
except UnexpectedResponse:
    print("Collection already exists, skipping creation")

print("Collection 'movies' created!")
print(f"  Vector size: 384 dimensions")
print(f"  Distance metric: Cosine Similarity")

Collection already exists, skipping creation
Collection 'movies' created!
  Vector size: 384 dimensions
  Distance metric: Cosine Similarity


In [25]:
# 10 movies with title, year, genre, and description
# The DESCRIPTION is what we'll embed â€” it captures the movie's meaning
movies = [
    {
        "id": 1,
        "title": "The Matrix",
        "year": 1999,
        "genre": "sci-fi",
        "description": "A computer hacker discovers that reality is a simulation created by machines to enslave humanity.",
    },
    {
        "id": 2,
        "title": "Inception",
        "year": 2010,
        "genre": "sci-fi",
        "description": "A skilled thief who steals corporate secrets through dream-sharing technology is given a chance to erase his criminal record.",
    },
    {
        "id": 3,
        "title": "The Godfather",
        "year": 1972,
        "genre": "crime",
        "description": "The aging patriarch of an organized crime dynasty transfers control of his empire to his reluctant youngest son.",
    },
    {
        "id": 4,
        "title": "Pulp Fiction",
        "year": 1994,
        "genre": "crime",
        "description": "Interconnected stories of criminals, a boxer, and a pair of bandits unfold in non-linear fashion in Los Angeles.",
    },
    {
        "id": 5,
        "title": "The Shawshank Redemption",
        "year": 1994,
        "genre": "drama",
        "description": "A banker sentenced to life in prison forms a transformative friendship with a fellow inmate over two decades.",
    },
    {
        "id": 6,
        "title": "Forrest Gump",
        "year": 1994,
        "genre": "drama",
        "description": "A man with a low IQ but good intentions witnesses and unwittingly influences several major historical events in America.",
    },
    {
        "id": 7,
        "title": "The Dark Knight",
        "year": 2008,
        "genre": "action",
        "description": "Batman faces the Joker, a criminal mastermind who wants to plunge Gotham City into anarchy and chaos.",
    },
    {
        "id": 8,
        "title": "Interstellar",
        "year": 2014,
        "genre": "sci-fi",
        "description": "A team of astronauts travels through a wormhole near Saturn in search of a new habitable planet for humanity.",
    },
    {
        "id": 9,
        "title": "Titanic",
        "year": 1997,
        "genre": "romance",
        "description": "A young couple from different social classes fall in love aboard the ill-fated maiden voyage of the RMS Titanic.",
    },
    {
        "id": 10,
        "title": "The Lion King",
        "year": 1994,
        "genre": "animation",
        "description": "A young lion prince flees his kingdom after the murder of his father and learns the true meaning of responsibility and bravery.",
    },
]

descriptions = [movie["description"] for movie in movies]
collections = qdrant_client.get_collections()
print(collections)



descriptions_embeddings = model.encode(descriptions)

for i ,movie in enumerate(descriptions_embeddings):
     print(f"{movie}<25{descriptions_embeddings[i][0]}, {movie}<25{descriptions_embeddings[i][1]}>")


ResponseHandlingException: [Errno 11001] getaddrinfo failed

In [10]:
#Adding all movies to Qdrant
from qdrant_client.models import PointStruct

points = [] #Each point is a vector in our database

for i,movie in enumerate(movies):
   point = PointStruct(
       id = movie["id"],
    vector = descriptions_embeddings[i].tolist(),
    payload = {
        "title": movie["title"],
        "year": movie["year"],
        "genre": movie["genre"],
        "description": movie["description"],
    },)

   points.append(point)


#insert data
qdrant_client.upsert(
    collection_name="movies",
    points=points,
)



#make year and genre searchable,genre match with exact word while year as integer


qdrant_client.create_payload_index(
    collection_name="movies",
    field_name="genre",
    field_schema="keyword",  # "keyword" for exact text matching
)

qdrant_client.create_payload_index(
    collection_name="movies",
    field_name="year",
    field_schema="integer",  # "integer" for numeric fields
)

#payload is data about vector(metadat)


# Verify the upsert worked
info = qdrant_client.get_collection("movies")
print(f"Successfully upserted {info.points_count} movies into Qdrant!")





Successfully upserted 10 movies into Qdrant!


***We Actually Match vectors ,when they matche we return their textual data ibstaed of vectors***

In [26]:
#Send query to database for searching matching vectors

query = "a movie about space travel and exploring other planets"

query_embeddings = model.encode(query)


--- Start of Task ---
Hello, world!
--- End of Task ---
